## **Implementing Django Channels**

[Refer_To_Docs](https://channels.readthedocs.io/en/latest/installation.html)

<hr>

### **Step 1: Install Django Channels**

```bash
python -m pip install -U 'channels[daphne]'
```

This will install Django Channels along with Daphne, the ASGI server.


## **Step 2: Update Django Settings**

In your `settings.py` file, add `'channels'` to your `INSTALLED_APPS` and set the ASGI application:

```python
INSTALLED_APPS = [
    ...,
    'channels',
]
ASGI_APPLICATION = 'your_project_name.asgi.application'
```

## **Step 3: Create ASGI Configuration**

Create an `asgi.py` file in your project directory (if it doesn't already exist) and configure it as follows:

```python
import os
from django.core.asgi import get_asgi_application
from channels.routing import ProtocolTypeRouter, URLRouter
from channels.auth import AuthMiddlewareStack
import your_app_name.routing

application = ProtocolTypeRouter({
    "http": get_asgi_application(),
    "websocket": AuthMiddlewareStack(
        URLRouter(
            your_app_name.routing.websocket_urlpatterns
        )
    ),
})
```

## **Step 4: Define Routing**

Create a `routing.py` file in your app directory and define your WebSocket URL patterns:

```python
from django.urls import re_path
from . import consumers

websocket_urlpatterns = [
    re_path(r'ws/some_path/$', consumers.MyConsumer.as_asgi()),
]
```

## **Step 5: Create Consumers**

Create a `consumers.py` file in your app directory and define your WebSocket consumer:

```python
from channels.generic.websocket import AsyncWebsocketConsumer
import json
class MyConsumer(AsyncWebsocketConsumer):
    async def connect(self):
        await self.accept()

    async def disconnect(self, close_code):
        pass

    async def receive(self, text_data):
        data = json.loads(text_data)
        # Handle the received data
        await self.send(text_data=json.dumps({
            'message': 'Message received!'
        }))
```

## **Step 6: Run the ASGI Server**

Run the Daphne server to serve your application:

```bash
daphne -p 8000 your_project_name.asgi:application
```

This will start the server on port 8000. You can now connect to your WebSocket endpoint at `ws://localhost:8000/ws/some_path/`.

<hr>


## **Notes**

- For each `Socket` connection, new layers are created. We can access these layers using `channel_layer`. This `channel_layer` remains persistent throughout the connection.

- We can create `Groups` to manage multiple connections together. `Group` is collection of channels that we can send messages to all at once.


## **Channel Layer**

Since, we've configured the `ASGI` application, and used `InMemoryChannelLayer` as our default channel layer, we can now use it in our consumers to send and receive messages.

**External Access to Channel Layer**

[Refer_To_Docs](https://channels.readthedocs.io/en/latest/topics/channel_layers.html#using-outside-of-consumers)

We can access this `Channel Layer` outside of consumers as well. For example, in a Django view or a management command.

We can access the channel layer using:

```python
from channels.layers import get_channel_layer
channel_layer = get_channel_layer()
```

And, if we want to send a message to a specific channel, we can do so using:

```python
await channel_layer.send('channel_name', {
    'type': 'chat.message',
    'message': 'Hello, World!'
})
```

**Internal Access to Channel Layer in Consumers**

Inside a consumer, we can access the channel layer using `self.channel_layer`. For example, to send a message to a group:

```python
await self.channel_layer.group_send(
    'group_name',
    {
        'type': 'chat.message',
        'message': 'Hello, Group!'
    }
)
```

## **Extra Notes**

### **Limiting Channel Layer Size per Group**

### **Broadcasting Messages to All Connected Clients**

To braodcast messages to all connected clients, you can create a group that includes all clients and send messages to that group. Here's how you can do it:

```python
# In your consumer
class MyConsumer(AsyncWebsocketConsumer):
    async def connect(self):
        await self.channel_layer.group_add(
            "broadcast_group",
            self.channel_name
        )
        await self.accept()
    async def disconnect(self, close_code):
        await self.channel_layer.group_discard(
            "broadcast_group",
            self.channel_name
        )
    async def receive(self, text_data):
        data = json.loads(text_data)
        # Broadcast the message to all clients in the group
        await self.channel_layer.group_send(
            "broadcast_group",
            {
                'type': 'chat.message',
                'message': data['message']
            }
        )
    async def chat_message(self, event):
        message = event['message']
        await self.send(text_data=json.dumps({
            'message': message
        }))
```

<hr>
